# CAREPILOT — SLA Breach Prediction Benchmark

*Chapter 2 — État de l'art et benchmarking IA · CAREPILOT operational AI*

Task 4 — proactive prediction of SLA violation risk before ticket resolution.

## 1. Objective

Predict whether an operational ITSM ticket is likely to **violate SLA** before resolution (`sla_breached`). This enables proactive prioritization, supervisor alerts, and queue balancing — complementing escalation (Task 1), transfer (Task 2), and action recommendation (Task 3).

## 2. Problem Type

- **Binary classification**
- **Target:** `sla_breached` (0 = within SLA, 1 = breach)
- **Positive class rate:** ~31.8% on the training corpus
- **Primary metrics:** F1-score, Recall, ROC-AUC (missing risky tickets is operationally costly)

## 3. Dataset Description

| Property | Value |
| --- | --- |
| Source | `ml_training.historical_tickets` |
| Total tickets | 14,000 |
| Historical actions | 32,766 |
| Historical resolutions | 8,233 |
| Train / validation / test | 9,800 / 2,100 / 2,100 |
| SLA breached (positive) | 31.82% |
| Missing description | 7.96% |
| Missing resolution time | 41.19% |

SLA breach is **not perfectly correlated** with priority, escalation, or transfer — the model must learn composite operational signals (queue load, ticket age, perimeter, complexity).

## 4. Preprocessing

1. **Text normalization** on title and description
2. **TF-IDF** vectorization (unigrams + bigrams, English stopwords)
3. **Categorical one-hot encoding** — perimeter, priority, incident category, environment type
4. **Numeric features** — ticket age, resolution time, queue load, store size, operational complexity
5. **Missing-value handling** — median imputation + explicit missingness indicator columns
6. **Stratified split** — 70% train / 15% validation / 15% test (`random_state=20260525`)

Priority is retained as a feature but SLA breach remains a distinct operational outcome.

## 5. Models Compared

1. **Logistic Regression** — transparent linear baseline; moderate recall, fast inference
2. **Random Forest** — nonlinear ensemble capturing perimeter and queue interactions
3. **XGBoost** — regularized gradient boosting; primary production candidate

This benchmark compares three supervised classifiers to select the most suitable SLA risk model for CAREPILOT integration.

## 6. Evaluation Metrics

- **Accuracy** — overall correctness (misleading under class imbalance)
- **Precision / Recall / F1** — per-class and aggregate; **Recall emphasized** for breach class
- **ROC-AUC** — ranking quality across thresholds
- **5-fold CV accuracy** — training-set stability (not used for final selection)
- **Training time (s)** and **Inference time (ms/row)** — operational deployability

## 7. Training Pipeline

```
historical_tickets → preprocessing (TF-IDF + encoding + scaling)
        → stratified split → train LogisticRegression / RandomForest / XGBoost
        → evaluate on held-out test (2,100 tickets)
        → confusion matrix, ROC, feature importance
```

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, Markdown, display

NOTEBOOK_DIR = Path.cwd()
for parent in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    if (parent / "ml" / "artifacts").exists():
        ROOT = parent
        break
else:
    raise FileNotFoundError("Project root not found")

SHARED = ROOT / "ml" / "notebooks" / "_shared"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from benchmark_notebook_helpers import display_plot, display_plots, find_project_root, plot_results_table

ROOT = find_project_root()
ARTIFACTS = ROOT / "ml" / "artifacts"
PLOTS = ROOT / "ml" / "plots"
REPORTS = ROOT / "ml" / "reports"
BENCHMARKS = ROOT / "ml" / "benchmarks"

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except NameError:
    pass

RUN_BENCHMARK = False


In [ ]:
script = BENCHMARKS / "sla_prediction" / "benchmark.py"
if RUN_BENCHMARK:
    subprocess.run([sys.executable, str(script), "--project-root", str(ROOT)], check=True)
else:
    print("Loading saved SLA benchmark artifacts.")
    print(f"  python {script} --project-root {ROOT}")


## 8. Benchmark Execution — Load Results

In [ ]:
metrics_path = ARTIFACTS / "03_sla_prediction_benchmark_metrics.json"
payload = json.loads(metrics_path.read_text(encoding="utf-8"))
ds = payload["dataset"]

display(Markdown("### Dataset statistics"))
display(
    pd.DataFrame(
        {
            "Metric": [
                "Total tickets",
                "SLA breached rate",
                "Train / val / test",
                "Missing description",
                "Missing resolution time",
            ],
            "Value": [
                ds["total_tickets"],
                f"{ds['class_distribution_pct']['1']:.2f}%",
                f"{ds['split_sizes']['train']} / {ds['split_sizes']['validation']} / {ds['split_sizes']['test']}",
                f"{ds['missing_description_pct']:.2f}%",
                f"{ds['missing_resolution_time_pct']:.2f}%",
            ],
        }
    )
)

results_df = pd.DataFrame(payload["results"])
display(Markdown("### Model comparison (held-out test split)"))
display(results_df)


## 9. Benchmark Results — Report Table

In [ ]:
figure_df = pd.DataFrame(payload["results"]).copy()
score_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
for col in score_cols:
    if col in figure_df.columns:
        figure_df[col] = figure_df[col].map(lambda x: f"{float(x):.3f}")

png_path = PLOTS / "03_sla_benchmark_results_table.png"
plot_results_table(
    figure_df[["Model", *score_cols]],
    title="CarePilot SLA Breach Prediction Benchmark",
    footer="Held-out test split · Target: sla_breached · Dataset: ml_training.historical_tickets (14,000 tickets)",
    output_path=png_path,
    col_widths=[0.22, 0.12, 0.13, 0.12, 0.10, 0.13],
)
display_plot(png_path)


## 10. Visualizations

In [ ]:
display_plots(
    [
        PLOTS / "03_sla_distribution.png",
        PLOTS / "03_priority_vs_sla.png",
        PLOTS / "03_perimeter_vs_sla.png",
        PLOTS / "03_sla_correlation_heatmap.png",
        PLOTS / "03_sla_model_comparison.png",
        PLOTS / "03_sla_roc_curves.png",
        PLOTS / "03_sla_precision_recall.png",
        PLOTS / "03_sla_f1_score.png",
    ]
)


In [ ]:
cm = payload.get("confusion_matrices", {})
if cm:
    fig, axes = plt.subplots(1, len(cm), figsize=(5 * len(cm), 4.2))
    if len(cm) == 1:
        axes = [axes]
    for ax, (model_name, matrix) in zip(axes, cm.items()):
        data = pd.DataFrame(matrix, index=["Actual: No breach", "Actual: SLA breach"], columns=["Pred: No", "Pred: Yes"])
        im = ax.imshow(data.values, cmap="Blues")
        ax.set_title(model_name)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(matrix[i][j]), ha="center", va="center", color="#111827")
        ax.set_xticks([0, 1], ["No", "Yes"])
        ax.set_yticks([0, 1], ["No", "Yes"])
    fig.suptitle("Confusion Matrices — SLA Breach Prediction", y=1.02)
    fig.tight_layout()
    plt.show()


## 11. Interpretation

| Model | F1 | Recall (breach) | ROC-AUC | Role |
| --- | --- | --- | --- | --- |
| Logistic Regression | 0.692 | 0.807 | 0.855 | Explainable baseline; high recall but lower precision |
| Random Forest | 0.677 | 0.707 | 0.847 | Nonlinear effects; balanced but weaker than XGBoost |
| **XGBoost** | **0.811** | **0.889** | **0.952** | **Best operational trade-off** |

- **Logistic Regression** achieves high recall (~81%) but lower precision — useful for auditability, not final routing.
- **Random Forest** captures perimeter/queue nonlinearities with moderate performance.
- **XGBoost** maximizes F1 and recall while maintaining strong precision (~75%) — best for proactive SLA supervision.

Missing resolution-time signals (41% of tickets) and store-level queue dynamics create irreducible ambiguity — scores remain in realistic enterprise bands, not near-perfect.

## 12. Selected Model

**Selected for CAREPILOT: XGBoost**

- Highest F1 (0.811) and recall (0.889) on held-out test split
- Production artifact: `ml/artifacts/models/xgboost_sla.joblib`
- Inference: ~0.09 ms/row — suitable for real-time ticket scoring

## 13. Limitations

- Resolution-time features may be unavailable at ticket creation in live workflows
- SLA contract definitions vary by perimeter and store type
- Historical breach labels reflect past operations, not future policy changes
- Threshold tuning required for HIGH/MEDIUM/LOW risk bands in production
- Model should be monitored as queue patterns and ticket volumes evolve

## 14. Generated Artifacts

- `ml/artifacts/03_sla_prediction_benchmark_metrics.json`
- `ml/artifacts/models/xgboost_sla.joblib`
- `ml/plots/03_sla_benchmark_results_table.png`
- `ml/plots/03_sla_distribution.png`, `03_sla_model_comparison.png`, `03_sla_roc_curves.png`
- `ml/reports/03_sla_prediction_benchmark_summary.md`